# Selection function verification

Verify the forward-modeled C27 selection by comparing the analytical
selection probability
$$P(\text{sel} \mid d, \sigma) = \Phi\!\left(\frac{-5\log_{10}(\pi_\text{cut}) - 5\log_{10} d}{\sigma}\right)$$
against Monte Carlo forward simulation: draw $d$, draw mock
$m_W^H \sim \mathcal{N}(M_\text{pred} + \mu(d),\, \sigma)$, compute
$\hat\pi$ from the mock magnitude, and cut on $\hat\pi > 0.8$.

The $M_\text{pred}$ terms cancel between the selection threshold and the
predicted apparent magnitude, so $P(\text{sel}\mid d)$ depends only on $d$
and $\sigma = \sqrt{\sigma_\text{mW}^2 + \sigma_\text{int}^2}$.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.integrate import simpson as integrate_simpson

from candel import load_config
from candel.pvdata import CepheidData, to_mwcepheids_config

%matplotlib inline
plt.rcParams.update({"font.size": 13})

In [ ]:
config = to_mwcepheids_config(load_config("../../scripts/runs/configs/config_MWCepheids_paper.toml", replace_los_prior=False))
data = CepheidData(config)
print(data)

# Reference parameters (posterior modes from converged MCMC)
M_H_1 = -6.13
b_W = -3.62
Z_W = -0.18
sigma_int = 0.09
delta_pi = 0.056

d_min, d_max = 0.1, 15.0
pi_cut = 0.8

# Predicted absolute magnitude per star
M_pred = M_H_1 + b_W * (np.array(data.logP) - 1.0) + Z_W * np.array(data.FeH)

In [ ]:
def log_likelihood(d, star_idx):
    """Log L(data_i | d, theta), vectorized over d."""
    mu = 5.0 * np.log10(d) + 10.0
    mW_pred = M_pred[star_idx] + mu
    mW_std = np.sqrt(float(data.mW_H_err[star_idx])**2 + sigma_int**2)
    ll = norm.logpdf(float(data.mW_H[star_idx]), loc=mW_pred, scale=mW_std)

    if bool(data.has_edr3[star_idx]):
        pi_model = 1.0 / d + delta_pi
        ll = ll + norm.logpdf(
            float(data.pi_EDR3[star_idx]), loc=pi_model,
            scale=float(data.pi_EDR3_err[star_idx]))

    return ll


def sample_d_prior(n, rng):
    """Draw d ~ p(d) propto d^2 on [d_min, d_max] via inverse CDF."""
    u = rng.uniform(0, 1, size=n)
    return (u * (d_max**3 - d_min**3) + d_min**3) ** (1.0 / 3.0)


def normalised_posterior(log_unnorm, d_grid):
    """Exponentiate and normalise a log-posterior on a grid."""
    f = np.exp(log_unnorm - log_unnorm.max())
    f /= integrate_simpson(f, x=d_grid)
    return f


def selection_prob(d, sigma):
    """P(pi_pred > pi_cut | d, sigma) = Phi(arg)."""
    arg = (-5.0 * np.log10(pi_cut) - 5.0 * np.log10(d)) / sigma
    return norm.cdf(arg)


def log_selection_prob(d, sigma):
    """Log P(sel | d, sigma), numerically stable."""
    arg = (-5.0 * np.log10(pi_cut) - 5.0 * np.log10(d)) / sigma
    return norm.logcdf(arg)

## 1. Selection probability $P(\text{sel} \mid d)$

In the forward model, $m_W^\text{obs} \sim \mathcal{N}(M_\text{pred} + \mu(d),\, \sigma)$.
The selection $\hat\pi > 0.8$ is equivalent to $m_W^\text{obs} < M_\text{pred} + 10 - 5\log_{10}(0.8)$.
The $M_\text{pred}$ terms cancel:
$$P(\text{sel} \mid d, \sigma) = \Phi\!\left(\frac{-5\log_{10}(0.8\, d)}{\sigma}\right)$$

This is a smooth step from 1 to 0 centred at $d = 1/\pi_\text{cut} = 1.25$ kpc,
broadened by $\sigma$.

In [ ]:
d_grid = np.linspace(d_min, d_max, 2001)

# Pick example stars: one C22, two C27 (different sigmas)
idx_c22 = int(np.where(np.array(data.is_c22))[0][0])
c27_indices = np.where(np.array(data.is_c27))[0]
idx_c27_0 = int(c27_indices[0])
idx_c27_1 = int(c27_indices[10])
examples = [idx_c22, idx_c27_0, idx_c27_1]
labels = [f"C22: {data.names[i]}" if bool(data.is_c22[i])
          else f"C27: {data.names[i]}" for i in examples]

fig, ax = plt.subplots(figsize=(7, 4))
for idx, label in zip(examples, labels):
    sigma = np.sqrt(float(data.mW_H_err[idx])**2 + sigma_int**2)
    P_sel = selection_prob(d_grid, sigma)
    ax.plot(d_grid, P_sel, lw=2,
            label=f"{label} ($\\sigma$ = {sigma:.3f})")

ax.axvline(1.0 / pi_cut, color="grey", ls="--", lw=1,
           label=f"$1/\\pi_\\mathrm{{cut}}$ = {1/pi_cut:.2f} kpc")
ax.set_xlabel("$d$ [kpc]")
ax.set_ylabel("$P(\\mathrm{sel} \\mid d)$")
ax.set_xlim(0, 4)
ax.legend(fontsize=10)
ax.set_title("Forward-modeled C27 selection probability")
plt.tight_layout()
plt.show()

## 2. Verify $P(\text{sel} \mid d)$ against MC forward simulation

For each $d_k$ drawn from the prior:
1. Compute $m_W^\text{pred} = M_\text{pred} + 5\log_{10} d_k + 10$
2. Draw $m_W^\text{mock} \sim \mathcal{N}(m_W^\text{pred}, \sigma)$
3. Compute $\hat\pi = 10^{-0.2(m_W^\text{mock} - M_\text{pred} - 10)}$
4. Accept if $\hat\pi > 0.8$

The acceptance fraction in distance bins should match $P(\text{sel}\mid d)$.

In [ ]:
rng = np.random.default_rng(42)
n_mc = 2_000_000

fig, axes = plt.subplots(1, len(examples), figsize=(5 * len(examples), 4))
for ax, idx, label in zip(axes, examples, labels):
    sigma = np.sqrt(float(data.mW_H_err[idx])**2 + sigma_int**2)

    # Draw d from prior
    d_mc = sample_d_prior(n_mc, rng)

    # Forward model: draw mock mW_obs
    mW_pred_mc = M_pred[idx] + 5.0 * np.log10(d_mc) + 10.0
    mW_mock = rng.normal(mW_pred_mc, sigma)

    # Compute pi_pred from mock mW and check selection
    pi_pred_mock = 10.0 ** (-0.2 * (mW_mock - M_pred[idx] - 10.0))
    selected = pi_pred_mock > pi_cut

    # Binned acceptance fraction
    n_bins = 80
    d_edges = np.linspace(0, 4.0, n_bins + 1)
    d_centres = 0.5 * (d_edges[:-1] + d_edges[1:])
    counts_all, _ = np.histogram(d_mc, bins=d_edges)
    counts_sel, _ = np.histogram(d_mc[selected], bins=d_edges)
    frac = np.where(counts_all > 0, counts_sel / counts_all, np.nan)

    # Analytical
    d_fine = np.linspace(0.01, 4.0, 500)
    P_sel = selection_prob(d_fine, sigma)

    ax.plot(d_fine, P_sel, "k-", lw=2, label="Analytical $\\Phi$")
    ax.scatter(d_centres, frac, s=12, color="C0", zorder=3,
              label="MC acceptance")
    ax.set_xlabel("$d$ [kpc]")
    ax.set_ylabel("$P(\\mathrm{sel} \\mid d)$")
    ax.set_title(label)
    ax.legend(fontsize=9)

fig.suptitle("Selection probability: analytical vs MC forward simulation",
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 3. Posterior on $d$ with selection

The posterior under selection is
$$p(d \mid \text{data}, \theta, \text{sel}) \propto
  \mathcal{L}(\text{data} \mid d, \theta)\; d^2\; P(\text{sel} \mid d, \sigma)$$

Compare the analytical grid calculation to MC: draw $d$ from prior,
forward-simulate and cut on $\hat\pi > 0.8$, weight accepted samples by
$\mathcal{L}(\text{real data} \mid d, \theta)$.

In [ ]:
fig, axes = plt.subplots(1, len(examples), figsize=(5 * len(examples), 4))
for ax, idx, label in zip(axes, examples, labels):
    sigma = np.sqrt(float(data.mW_H_err[idx])**2 + sigma_int**2)
    is_c27 = bool(data.is_c27[idx])

    # Analytical: no selection
    log_post0 = 2.0 * np.log(d_grid) + log_likelihood(d_grid, idx)
    post0 = normalised_posterior(log_post0, d_grid)

    # Analytical: with selection (C27 only)
    if is_c27:
        log_post1 = log_post0 + log_selection_prob(d_grid, sigma)
    else:
        log_post1 = log_post0.copy()
    post1 = normalised_posterior(log_post1, d_grid)

    # MC: forward simulate, cut, weight by real-data likelihood
    d_mc = sample_d_prior(n_mc, rng)
    if is_c27:
        mW_pred_mc = M_pred[idx] + 5.0 * np.log10(d_mc) + 10.0
        mW_mock = rng.normal(mW_pred_mc, sigma)
        pi_pred_mock = 10.0 ** (-0.2 * (mW_mock - M_pred[idx] - 10.0))
        selected = pi_pred_mock > pi_cut
        d_sel = d_mc[selected]
    else:
        d_sel = d_mc  # no cut for C22

    log_w = log_likelihood(d_sel, idx)
    w = np.exp(log_w - log_w.max())
    w /= w.sum()

    ax.plot(d_grid, post0, "k--", lw=1.5, label="No selection")
    ax.plot(d_grid, post1, "k-", lw=2, label="With selection")
    ax.hist(d_sel, bins=150, weights=w, density=True,
            alpha=0.5, color="C0", label="MC (fwd sim + cut)")
    ax.set_xlabel("$d$ [kpc]")
    ax.set_ylabel(r"$p(d \mid \mathrm{data}, \theta, \mathrm{sel})$")
    ax.set_title(label)
    ax.legend(fontsize=9)

fig.suptitle("Posterior on $d$: analytical vs MC with forward-modeled selection",
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4. Normalization integral $Z_S$

Validate the normalization
$$Z_S = \int d^2\, P(\text{sel} \mid d, \sigma)\, \mathrm{d}d$$
computed via Simpson's rule against MC: $Z_S = Z_\text{prior} \cdot \mathbb{E}_{d\sim p(d)}[P(\text{sel}\mid d)]$.

In [ ]:
Z_prior = (d_max**3 - d_min**3) / 3.0

print(f"{'star':>20s}  {'sigma':>8s}  {'Z_S (Simpson)':>14s}  {'Z_S (MC)':>14s}")
print("-" * 65)

for idx, label in zip(examples, labels):
    sigma = np.sqrt(float(data.mW_H_err[idx])**2 + sigma_int**2)

    # Simpson
    integrand = d_grid**2 * selection_prob(d_grid, sigma)
    Z_S_simp = integrate_simpson(integrand, x=d_grid)

    # MC: E_{p(d)}[P(sel | d)] * Z_prior
    d_mc = sample_d_prior(5_000_000, rng)
    P_sel_mc = selection_prob(d_mc, sigma)
    Z_S_mc = np.mean(P_sel_mc) * Z_prior

    print(f"{label:>20s}  {sigma:8.4f}  {Z_S_simp:14.6f}  {Z_S_mc:14.6f}")

## 5. Effect on the marginal likelihood over $M_{H,1}$

Compute the marginal likelihood
$$\ln p(\text{data}_i \mid \theta, \text{sel}) =
  \ln \int \mathcal{L}_i\, d^2\, P(\text{sel} \mid d)\, \mathrm{d}d - \ln Z_S$$
as a function of $M_{H,1}$.  Compare three cases:
1. No selection
2. Selection **without** normalization (wrong)
3. Selection **with** normalization (correct)

In [ ]:
def log_marginal_likelihood(M_H_1_val, star_idx, with_selection=False):
    """Log marginal likelihood for one star, marginalized over d.

    Returns (log_ml, log_Z_S).  If with_selection is False, log_Z_S = 0.
    """
    M_pred_i = (M_H_1_val
                + b_W * (float(data.logP[star_idx]) - 1.0)
                + Z_W * float(data.FeH[star_idx]))
    sigma = np.sqrt(float(data.mW_H_err[star_idx])**2 + sigma_int**2)
    mu = 5.0 * np.log10(d_grid) + 10.0
    mW_pred = M_pred_i + mu
    ll = norm.logpdf(float(data.mW_H[star_idx]), loc=mW_pred, scale=sigma)

    if bool(data.has_edr3[star_idx]):
        pi_model = 1.0 / d_grid + delta_pi
        ll = ll + norm.logpdf(
            float(data.pi_EDR3[star_idx]), loc=pi_model,
            scale=float(data.pi_EDR3_err[star_idx]))

    log_integrand = 2.0 * np.log(d_grid) + ll

    if with_selection and bool(data.is_c27[star_idx]):
        log_P = log_selection_prob(d_grid, sigma)
        log_integrand_sel = log_integrand + log_P

        # Z_S = int d^2 P(sel|d) dd
        integrand_Z = d_grid**2 * selection_prob(d_grid, sigma)
        log_Z_S = np.log(integrate_simpson(integrand_Z, x=d_grid))

        f = np.exp(log_integrand_sel - log_integrand_sel.max())
        log_ml = (np.log(integrate_simpson(f, x=d_grid))
                  + log_integrand_sel.max())
        return log_ml, log_Z_S

    f = np.exp(log_integrand - log_integrand.max())
    log_ml = np.log(integrate_simpson(f, x=d_grid)) + log_integrand.max()
    return log_ml, 0.0

In [ ]:
M_grid = np.linspace(-7.0, -5.0, 200)

# Only plot C27 stars where the selection matters
c27_examples = [(i, l) for i, l in zip(examples, labels) if bool(data.is_c27[i])]

fig, axes = plt.subplots(1, max(len(c27_examples), 1),
                         figsize=(6 * max(len(c27_examples), 1), 4),
                         squeeze=False)
axes = axes.ravel()

for ax, (idx, label) in zip(axes, c27_examples):
    lml_no = np.array([log_marginal_likelihood(m, idx, False)[0]
                       for m in M_grid])
    lml_sel = np.array([log_marginal_likelihood(m, idx, True)
                        for m in M_grid])
    lml_sel_no_norm = lml_sel[:, 0]
    lml_sel_norm = lml_sel[:, 0] - lml_sel[:, 1]

    ax.plot(M_grid, lml_no - lml_no.max(),
            "k-", lw=2, label="No selection")
    ax.plot(M_grid, lml_sel_no_norm - lml_sel_no_norm.max(),
            "r--", lw=2, label="Sel, no norm (wrong)")
    ax.plot(M_grid, lml_sel_norm - lml_sel_norm.max(),
            "C0-", lw=2, label="Sel + norm (correct)")
    ax.set_xlabel("$M_{H,1}$")
    ax.set_ylabel(r"$\Delta \ln p(\mathrm{data} \mid \theta, \mathrm{sel})$")
    ax.set_title(label)
    ax.legend(fontsize=9)

fig.suptitle("Marginal likelihood vs $M_{H,1}$ (C27 forward-modeled selection)",
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 6. MC validation of the marginal likelihood

Estimate the marginal likelihood via MC for a few values of $M_{H,1}$.

MC (with selection):
$$p(\text{data} \mid \theta, \text{sel})
  = \frac{\mathbb{E}_{d\sim p(d)}[\mathcal{L}\, P(\text{sel}\mid d)]}
         {\mathbb{E}_{d\sim p(d)}[P(\text{sel}\mid d)]}$$

In [ ]:
n_mc_ml = 2_000_000
M_test = np.array([-6.5, -6.13, -5.8])
log_Z_prior = np.log((d_max**3 - d_min**3) / 3.0)

print(f"{'M_H_1':>8s}  {'star':>20s}  {'analytical':>12s}  {'MC':>12s}  "
      f"{'anal(sel)':>12s}  {'MC(sel)':>12s}")
print("-" * 90)

for idx, label in zip(examples, labels):
    is_c27 = bool(data.is_c27[idx])
    sigma = np.sqrt(float(data.mW_H_err[idx])**2 + sigma_int**2)

    for m_val in M_test:
        lml_a, _ = log_marginal_likelihood(m_val, idx, False)
        lml_a_sel, lml_a_zs = log_marginal_likelihood(m_val, idx, True)
        lml_a_sel_corr = lml_a_sel - lml_a_zs

        # MC
        d_mc = sample_d_prior(n_mc_ml, rng)
        M_pred_i = (m_val
                    + b_W * (float(data.logP[idx]) - 1.0)
                    + Z_W * float(data.FeH[idx]))
        mu_mc = 5.0 * np.log10(d_mc) + 10.0
        mW_pred_mc = M_pred_i + mu_mc
        ll_mc = norm.logpdf(float(data.mW_H[idx]), loc=mW_pred_mc,
                            scale=sigma)
        if bool(data.has_edr3[idx]):
            pi_model_mc = 1.0 / d_mc + delta_pi
            ll_mc = ll_mc + norm.logpdf(
                float(data.pi_EDR3[idx]), loc=pi_model_mc,
                scale=float(data.pi_EDR3_err[idx]))

        ll_max = ll_mc.max()

        # No selection
        lml_mc = (np.log(np.mean(np.exp(ll_mc - ll_max)))
                  + ll_max + log_Z_prior)

        # With selection: E[L*P_sel] / E[P_sel]
        if is_c27:
            P_sel_mc = selection_prob(d_mc, sigma)
            LP = np.exp(ll_mc - ll_max) * P_sel_mc
            lml_mc_sel = (np.log(np.mean(LP)) + ll_max
                          - np.log(np.mean(P_sel_mc)))
        else:
            lml_mc_sel = lml_mc  # no C27 selection for C22 stars

        print(f"{m_val:8.2f}  {label:>20s}  {lml_a:12.4f}  {lml_mc:12.4f}  "
              f"{lml_a_sel_corr:12.4f}  {lml_mc_sel:12.4f}")

## Summary

The C27 selection $\hat\pi > 0.8$ is **distance-dependent** in the forward
model: at distance $d$, the probability that a mock $m_W^H$ passes the cut
is $\Phi((-5\log_{10}(0.8\,d)) / \sigma)$.  This acts as a smooth truncation
around $d \approx 1/0.8 = 1.25$ kpc.

Both the selection factor and its normalization $Z_S$ must be included
in the model.  The analytical $\Phi$ formula matches MC forward simulation
to high precision.